In [ ]:
# E22 flag kit: the E22 vLLM Speed Lab launch commands against the real vLLM 0.32.0 (CPU, pinned image)
import hashlib, io, json, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_e22_flags_kit_39ce1c4759cd.bin", "39ce1c4759cd6fa4e462774fa7b871d0d067069dff68fb646a827bcfb3cc53a8"
WH032 = "/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse"
OUT, K, T0 = Path("/kaggle/working"), Path("/tmp/e22_flags"), time.time()
REPORT = {"kit": KIT_NAME, "python": sys.version}
f = next(p for pat in (KIT_NAME, "*/" + KIT_NAME, "*/*/" + KIT_NAME, "*/*/*/" + KIT_NAME) for p in Path("/kaggle/input").glob(pat))
b = f.read_bytes(); assert hashlib.sha256(b).hexdigest() == KIT_SHA
zipfile.ZipFile(io.BytesIO(b)).extractall(K)
inst = next(iter(sorted(Path(WH032).rglob("install_wheelhouse.py"))), None)
assert inst is not None, "attach " + WH032
r = subprocess.run([sys.executable, str(inst), "/tmp/vllm-venv"], capture_output=True, text=True, timeout=2400)
REPORT["install"] = {"rc": r.returncode, "sec": round(time.time() - T0), "tail": (r.stdout + r.stderr)[-800:]}
vpy = next((l.split("=", 1)[1] for l in r.stdout.splitlines() if l.startswith("VENV_PYTHON=")), "/tmp/vllm-venv/bin/python")
r = subprocess.run([vpy, str(K / "e21_flagcheck.py"), str(K / "e22_flags.json"), "0.32.0"], capture_output=True, text=True, timeout=1800)
REPORT["flagcheck"] = {"rc": r.returncode, "out": (r.stdout + r.stderr)[-8000:],
                       "summary": next((json.loads(l[10:]) for l in r.stdout.splitlines() if l.startswith("FLAGCHECK ")), None)}
print("flagcheck:", json.dumps(REPORT["flagcheck"]["summary"])[:3000], flush=True)
r = subprocess.run([vpy, str(K / "e22_fieldcheck.py"), str(K / "e22_flags.json")], capture_output=True, text=True, timeout=900)
REPORT["fieldcheck"] = next((json.loads(l[11:]) for l in r.stdout.splitlines() if l.startswith("FIELDCHECK ")),
                            {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-3000:]})
print("fieldcheck:", json.dumps(REPORT["fieldcheck"])[:4000], flush=True)
REPORT["all_ok"] = bool((REPORT["flagcheck"]["summary"] or {}).get("all_ok")) and bool(REPORT["fieldcheck"].get("all_ok"))
REPORT["sec"] = round(time.time() - T0)
(OUT / "e22_flags_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
